# Session 12: Context Managers, JSON, Decorators and Generators

**Python Fundamentals | Session 12 of 14 | Guided learning: 30 minutes**

The Learning Assistant now needs to save and restore study activities. Follow **Predict â†’ Run â†’ Modify â†’ Verify â†’ Reflect**. This notebook uses only the Python standard library, temporary files and deterministic examples. It runs safely in Google Colab or VS Code.

## 1. Why Applications Need Persistent Data (2 minutes)

**CONCEPT:** A Python list lives in memory while the programme runs. A new process does not automatically receive that list. A file stores text that can be read later, while JSON is a portable text format for dictionaries and lists. Python objects are the values currently used by the programme.

**PREDICT:** If we create a list and close the programme, will the list automatically appear when we start a new programme? No. Data must be stored explicitly if we want to recover it later.

In [1]:
sessions = [
    {"topic": "Python", "duration": 45},
    {"topic": "Mathematics", "duration": 60},
]
print(sessions)
assert len(sessions) == 2
assert sessions[0]["topic"] == "Python"

[{'topic': 'Python', 'duration': 45}, {'topic': 'Mathematics', 'duration': 60}]


**MODIFY:** Add a third study dictionary to the list. It exists only in memory until a later file-writing step stores it.

**VERIFY:** In-memory data and saved JSON are different representations of the same information.

**REFLECT:** Why must an application explicitly save data before it can restore it?

## 2. Context Managers and Safe File Handling (4 minutes)

**CONCEPT:** `open()` creates a file connection. `with open(...) as file:` manages that resource and closes it when the block ends, even if a handled problem occurs. Modes are `"r"` for reading, `"w"` for writing (replacing existing content) and `"a"` for appending. UTF-8 supports a wide range of characters.

**PREDICT:** If a file is opened with `"w"` twice, will the second write follow the first text or replace it? It replaces the existing content. `"a"` adds to the end.

In [3]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    path = Path(directory) / "notes.txt"
    with open(path, "w", encoding="utf-8") as file:
        file.write("Python practice: 45 minutes")
    with open(path, "r", encoding="utf-8") as file:
        restored = file.read()
    print(restored)
    assert restored == "Python practice: 45 minutes"
    assert file.closed
assert not path.exists()


Python practice: 45 minutes


**DEBUG 1:** This flawed code leaves resource management unclear and uses `"w"` when the intention is to append:

```python
file = open(path, "w")
file.write("Another note")
```

Use a context manager and mode `"a"` for the repair. The temporary directory keeps the exercise self-contained.

In [4]:
with TemporaryDirectory() as directory:
    path = Path(directory) / "notes.txt"
    with open(path, "w", encoding="utf-8") as file:
        file.write("Python: 45 minutes\n")
    with open(path, "a", encoding="utf-8") as file:
        file.write("Mathematics: 60 minutes\n")
    with open(path, "r", encoding="utf-8") as file:
        notes = file.readlines()
    assert len(notes) == 2
    assert notes[1] == "Mathematics: 60 minutes\n"
    print(notes)

['Python: 45 minutes\n', 'Mathematics: 60 minutes\n']


**MODIFY:** Write two different study notes and read them back. **REFLECT:** Why is automatic closing safer than relying on a later manual `close()` call?

## 3. JSON: From Python Objects to Structured Text (4 minutes)

**CONCEPT:** Serialization converts a Python value to JSON text; deserialization converts JSON text back to a Python value. Python dictionaries/lists correspond to JSON objects/arrays. Python `True`, `False` and `None` become JSON `true`, `false` and `null`. JSON object keys are strings.

| Python | JSON |
| --- | --- |
| `dict` | object |
| `list` | array |
| `str`, `int`, `float` | string, number |
| `True`, `False`, `None` | `true`, `false`, `null` |

**PREDICT:** What Python type will `json.loads(json_text)` return for one session? A dictionary.

In [ ]:
import json

session = {
    "type": "G",
    "topic": "Mathematics",
    "duration": 60,
    "mentor": "Alex",
}
json_text = json.dumps(session, ensure_ascii=False)
restored_session = json.loads(json_text)
print(json_text)
print(type(restored_session).__name__, restored_session)
assert isinstance(restored_session, dict)
assert restored_session["duration"] == 60
assert restored_session["mentor"] == "Alex"

**MODIFY:** Add `"completed": True` and a Unicode topic such as `"Ð£ÐºÑ€Ð°Ñ—Ð½ÑÑŒÐºÐ°"`. Serialize and deserialize again, then verify both fields survive.

**VERIFY:** `dumps()` returns a string; `loads()` accepts that string and returns a Python value. JSON text is not identical to Python syntax, even though the structures correspond.

**REFLECT:** Why is a round trip useful before saving a larger application state?

## 4. Saving and Loading JSON Files (4 minutes)

**CONCEPT:** `json.dump(data, file)` writes JSON to a file-like object; `json.load(file)` reads JSON from one. In contrast, `dumps()` and `loads()` work with strings. The order of a JSON list is preserved.

**PREDICT:** What happens when `"w"` mode is used twice for the same JSON file? The second write replaces the first file contents.

In [ ]:
sessions_to_save = [
    {"type": "I", "topic": "Python", "duration": 45, "mentor": ""},
{"type": "G", "topic": "Mathematics", "duration": 60, "mentor": "Alex"},
    {"type": "I", "topic": "English", "duration": 20, "mentor": ""},
]

with TemporaryDirectory() as directory:
    path = Path(directory) / "sessions.json"
    with open(path, "w", encoding="utf-8") as file:
        json.dump(sessions_to_save, file, indent=2, ensure_ascii=False)
    with open(path, "r", encoding="utf-8") as file:
        restored_sessions = json.load(file)
    print(restored_sessions)
    assert len(restored_sessions) == 3
    assert restored_sessions[0]["topic"] == "Python"
    assert restored_sessions[1]["mentor"] == "Alex"
    assert [item["duration"] for item in restored_sessions] == [45, 60, 20]

**DEBUG 2:** This AI-generated code confuses a path with a file object:

```python
json.dump(sessions, "sessions.json")
```

`dump()` needs an opened writable file. Another common mistake is using `json.loads(file)` instead of `json.load(file)`. Repair both choices in the next example.

In [ ]:
with TemporaryDirectory() as directory:
    path = Path(directory) / "sessions.json"
    with open(path, "w", encoding="utf-8") as file:
        json.dump(sessions_to_save, file)
    with open(path, "r", encoding="utf-8") as file:
        repaired = json.load(file)
    assert repaired == sessions_to_save
    print("Correct file-based JSON round trip completed.")

**MODIFY:** Add one more session and verify that the restored count is four. **REFLECT:** Why is distinguishing strings from file-like objects important when choosing between `loads()` and `load()`?

## 5. Handling File and JSON Errors (3 minutes)

**CONCEPT:** A missing file raises `FileNotFoundError`; malformed JSON raises `json.JSONDecodeError`; invalid application data may raise `ValueError`. A missing file can reasonably mean that no archive exists yet. Corrupted JSON should not silently become an empty archive, because that could hide valuable data.

**PREDICT:** Which exception should `load_json_or_empty()` handle, and which should it allow to remain visible? Handle only `FileNotFoundError`; allow malformed JSON to signal a real problem.

In [ ]:
def load_json_or_empty(path):
    try:
        with open(path, "r", encoding="utf-8") as file:
            return json.load(file)
    except FileNotFoundError:
        return []

with TemporaryDirectory() as directory:
    missing_path = Path(directory) / "missing.json"
    assert load_json_or_empty(missing_path) == []
    valid_path = Path(directory) / "valid.json"
    valid_path.write_text("[{\"topic\": \"Python\"}]", encoding="utf-8")
    assert load_json_or_empty(valid_path)[0]["topic"] == "Python"
    broken_path = Path(directory) / "broken.json"
    broken_path.write_text("not JSON", encoding="utf-8")
    try:
        load_json_or_empty(broken_path)
    except json.JSONDecodeError:
        malformed_was_reported = True
    else:
        malformed_was_reported = False
    assert malformed_was_reported
print("Missing files are empty; malformed files are not silently accepted.")

**DEBUG 3:** This unsafe implementation hides every problem:

```python
def load_sessions(path):
    try:
        with open(path, encoding="utf-8") as file:
            return json.load(file)
    except:
        return []
```

Ask: which errors could it hide, and why might treating corrupted data as empty risk overwriting valuable sessions? The focused repair catches only `FileNotFoundError`.

## 6. Decorators: Adding Behaviour Around Functions (4 minutes)

**CONCEPT:** A decorator receives a function and returns a wrapped function. The `@announce_call` form is equivalent to assigning `calculate_total = announce_call(calculate_total)`. The wrapper must return the original result so the decorated function remains useful.

**PREDICT:** What prints first and last when the decorated calculation runs? The announcement comes first, then the completion message; the returned total is still available.

In [ ]:
from functools import wraps

def announce_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("Starting calculation")
        result = func(*args, **kwargs)
        print("Calculation completed")
        return result
    return wrapper

@announce_call
def calculate_total(minutes):
    return sum(minutes)

total = calculate_total([30, 45, 60])
assert total == 135
assert calculate_total.__name__ == "calculate_total"
print("Returned total:", total)

**DEBUG 4:** A flawed wrapper calls the function but forgets to return its result:

```python
def broken_announce(func):
    def wrapper(*args, **kwargs):
        print("Starting")
        func(*args, **kwargs)
    return wrapper
```

Predict the resulting value: it is `None`. The corrected wrapper returns `result`.

In [ ]:
def repaired_announce(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("Starting repaired calculation")
        result = func(*args, **kwargs)
        print("Finished repaired calculation")
        return result
    return wrapper

@repaired_announce
def repaired_total(minutes):
    return sum(minutes)

assert repaired_total([20, 25]) == 45
print("The repaired decorator preserved the result.")

**MODIFY:** Change the displayed messages while keeping the calculation result unchanged. **REFLECT:** What behaviour does the decorator add, and what behaviour must it preserve?

## 7. Generators and Lazy Evaluation (4 minutes)

**CONCEPT:** A generator function uses `yield` to produce one value at a time. Calling it returns a generator object; its body runs as a loop requests values. `return` finishes a function with one result, while `yield` pauses and resumes a sequence. This lazy evaluation can avoid building a large result list at once.

**PREDICT:** What type is `focused_sessions(sessions, 30)` before iteration? It is a generator object, not a list.

In [ ]:
def focused_sessions(sessions, minimum):
    for session in sessions:
        if session["duration"] >= minimum:
            yield session

study_sessions = [
    {"topic": "Python", "duration": 45},
    {"topic": "English", "duration": 20},
    {"topic": "Mathematics", "duration": 60},
]
focused = focused_sessions(study_sessions, 30)
print(type(focused).__name__)
focused_list = list(focused)
print(focused_list)
assert focused_list == [
    {"topic": "Python", "duration": 45},
    {"topic": "Mathematics", "duration": 60},
]
assert list(focused_sessions(study_sessions, 100)) == []

In [ ]:
one_at_a_time = focused_sessions(study_sessions, 30)
first = next(one_at_a_time)
second = next(one_at_a_time)
try:
    next(one_at_a_time)
except StopIteration:
    finished_safely = True
else:
    finished_safely = False
assert first["topic"] == "Python"
assert second["topic"] == "Mathematics"
assert finished_safely
print("The generator was consumed one value at a time.")

**MODIFY:** Change the threshold to `50`. Only Mathematics should remain. Remember that a generator is consumed as it is iterated; create a new generator for another pass.

**VERIFY:** `list(generator)` collects the remaining yielded values. A second `list()` on the same consumed generator is empty.

**REFLECT:** When might yielding one record at a time be more practical than constructing a large list?

## 8. Mini Integration: The Reliable Study Archive (3 minutes)

**CONCEPT:** The archive combines a context manager for safe files, JSON for persistence, a decorator for calculation announcements and a generator for focused-session selection.

**PREDICT:** With a minimum focused duration of 30 minutes, will the total be 105? Yes: 45 + 60.

In [ ]:
archive_sessions = [
    {"type": "I", "topic": "Python", "duration": 45, "mentor": ""},
    {"type": "G", "topic": "Mathematics", "duration": 60, "mentor": "Alex"},
    {"type": "I", "topic": "English", "duration": 20, "mentor": ""},
]

@announce_call
def focused_total(sessions, minimum):
    return sum(item["duration"] for item in focused_sessions(sessions, minimum))

with TemporaryDirectory() as directory:
    archive_path = Path(directory) / "study_archive.json"
    with open(archive_path, "w", encoding="utf-8") as file:
        json.dump(archive_sessions, file, indent=2)
    with open(archive_path, "r", encoding="utf-8") as file:
        restored_archive = json.load(file)
    focused = list(focused_sessions(restored_archive, 30))
    total_focused = focused_total(restored_archive, 30)
    assert len(restored_archive) == 3
    assert restored_archive[0]["topic"] == "Python"
    assert restored_archive[1]["mentor"] == "Alex"
    assert len(focused) == 2
    assert total_focused == 105
    print("Focused total:", total_focused)

## 9. Final Verification and Reflection (2 minutes)

**VERIFY:** The final checks confirm the essential techniques without requiring permanent files or network access.

In [ ]:
assert json.loads(json.dumps({"duration": 45}))["duration"] == 45
assert repaired_total([12]) == 12
assert list(focused_sessions([{"duration": 10}], 30)) == []
assert focused_total(archive_sessions, 50) == 60
with TemporaryDirectory() as directory:
    final_path = Path(directory) / "final.json"
    with open(final_path, "w", encoding="utf-8") as file:
        json.dump(archive_sessions, file)
    with open(final_path, encoding="utf-8") as file:
        assert len(json.load(file)) == 3
print("All Session 12 verification checks passed.")

**REFLECT:**

1. Why is `with open(...)` safer than manually managing a file? It closes the resource automatically.
2. What is the difference between `json.dump()` and `json.dumps()`? `dump()` writes to a file-like object; `dumps()` returns JSON text.
3. Why should malformed JSON not be silently replaced with an empty list? It could hide corruption and risk losing valuable data.
4. What does a decorator change about the way a function is called? It adds reusable behaviour around the call while preserving the result.
5. What is the main difference between `yield` and `return`? `yield` produces a sequence lazily; `return` finishes with a result.
6. Why is reading saved data back important? It verifies that persistence and restoration preserve the application data.

**Context managers manage resources. JSON preserves structured data. Decorators add reusable behaviour. Generators produce values when needed.**

These techniques prepare the Learning Assistant 12.0 Engineering Challenge.